# File

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv("ML_Ready_Extended_Geography.csv") # ETO YUNG GINAMIT KONG CSV YA CHINECK Q NA
X = df.drop('HIV_Status', axis=1)
y = df['HIV_Status']

# 1. First split to separate out the unseen TEST set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

# 2. Second split to separate Train and Validation from the remaining 80%
# 0.25 x 0.8 = 0.2 (So we end up with 60% Train, 20% Val, 20% Test)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, stratify=y_temp, test_size=0.25, random_state=42
)

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [4]:
df.head(10)

,Sex,Age_Group,Transmission,Healthcare_Access_Friction,Civil_Status,OFW_Status,Chemsex_Engagement,Alcohol_Sex_Risk,PrEP_Awareness,Transactional_Sex,STI_BBV_CoInfection_Any,HIV_Status
0,Female,<15,Male to Female Sex,2,Single,No,No,No,No,No TS,No,Reactive
1,Male,15-24,Male to Female Sex,2,Single,No,No,No,No,No TS,No,Reactive
2,Male,15-24,Male to Male Sex,2,Single,No,No,No,Yes,No TS,Yes,Reactive
3,Male,15-24,Male to Male Sex,2,Single,No,No,No,No,No TS,Yes,Reactive
4,Male,15-24,Male to Male Sex,2,Single,No,Yes,No,Yes,No TS,No,Reactive
5,Male,15-24,Male to Male/Female Sex,2,Single,No,No,Yes,Yes,Both,No,Reactive
6,Male,25-34,Male to Female Sex,2,Common-Law,No,No,No,No,No TS,No,Reactive
7,Male,25-34,Male to Male Sex,2,Single,No,No,No,Yes,No TS,No,Reactive
8,Male,25-34,Male to Male Sex,2,Single,No,No,No,No,No TS,No,Reactive
9,Male,25-34,Male to Male Sex,2,Single,No,No,Yes,Yes,Paid for sex,Yes,Reactive


# Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# list num and cat
numeric_features = []
categorical_features = ['Sex','Age_Group','Transmission',
                        'Healthcare_Access_Friction','Civil_Status','OFW_Status','Chemsex_Engagement',
                        'Alcohol_Sex_Risk','PrEP_Awareness','Transactional_Sex','STI_BBV_CoInfection_Any']

# 2. proprocessor
preprocessor = ColumnTransformer(
    transformers=[
        # Apply StandardScaler to numeric columns
        ('num', StandardScaler(), numeric_features),

        # Apply OneHotEncoder to categorical columns
        # handle_unknown='ignore' ensures the code won't crash if X_test has a category not seen in X_train
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='passthrough' # Keep any other columns not listed above (or use 'drop')
)

# 3. Fit and Transform
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

X_val_processed = preprocessor.transform(X_val) 


# Convert into DataFrame AND preserve the original index
X_train_processed = pd.DataFrame(X_train_processed, columns=preprocessor.get_feature_names_out(), index=X_train.index)
X_test_processed = pd.DataFrame(X_test_processed, columns=preprocessor.get_feature_names_out(), index=X_test.index)

#Convert validation set to DataFrame
X_val_processed = pd.DataFrame(
    X_val_processed,
    columns=preprocessor.get_feature_names_out(),
    index=X_val.index
)


# Dealing with y columns
mapping = {'Non-Reactive': 0, 'Reactive': 1}
y_train_processed = y_train.map(mapping)
y_test_processed = y_test.map(mapping)

# ADD THIS LINE: Map the validation labels!
y_val_processed = y_val.map(mapping)

 # MODEL

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, f1_score, average_precision_score
import xgboost as xgb
import numpy as np
# 1. Calculate the scale_pos_weight for your 1:10 imbalance
neg_count = (y_train_processed == 0).sum()
pos_count = (y_train_processed == 1).sum()
spw = neg_count / pos_count
print(f"Calculated scale_pos_weight for XGBoost: {spw:.2f}")

# 2. Initialize Base XGBoost
xgb_base = xgb.XGBClassifier(
    scale_pos_weight=spw,
    random_state=42,
    eval_metric='aucpr', # Tell it to optimize for Precision-Recall Area!
    n_jobs=-1
)

# 3. Define the Hyperparameter Grid
param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],         # Tree depth (lower prevents overfitting)
    'learning_rate': [0.01, 0.1]    # How fast it learns from mistakes
}

# 4. Setup Grid Search
grid_search_xgb = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid_xgb,
    scoring='average_precision',
    cv=5,
    verbose=1,
    n_jobs=-1
)

# 5. Train the Model!
print("\nTraining XGBoost...")
grid_search_xgb.fit(X_train_processed, y_train_processed)

best_xgb_model = grid_search_xgb.best_estimator_

# 6. Make Predictions
y_pred_xgb = best_xgb_model.predict(X_test_processed)
y_probs_xgb = best_xgb_model.predict_proba(X_test_processed)[:, 1]

# 7. Evaluate
print("\n--- XGBoost Final Report ---")
print(f"Best Parameters: {grid_search_xgb.best_params_}")
print(classification_report(y_test_processed, y_pred_xgb, target_names=['Non Reactive', 'Reactive']))
print(f"XGB F1-Score: {f1_score(y_test_processed, y_pred_xgb):.4f}")
print(f"XGB AUPRC: {average_precision_score(y_test_processed, y_probs_xgb):.4f}")

# Threshold Moving 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, 
    precision_score, 
    recall_score,
    precision_recall_curve
)

# ---------------------------------------------------------
# STEP 1: FIND THE BEST THRESHOLD ON THE *VALIDATION* SET
# ---------------------------------------------------------

# Get probabilities from the validation set
xgb_probs_val = best_xgb_model.predict_proba(X_val_processed)[:, 1]

# precision_recall_curve gives us all values efficiently
precisions, recalls, thresholds = precision_recall_curve(y_val_processed, xgb_probs_val)

# Calculate F1 scores for all thresholds mathematically (vectorized)
# We add 1e-8 to the denominator to prevent division by zero
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)

# Find the threshold that maximizes the F1 score
# Note: precision_recall_curve returns one less threshold than precision/recall values
best_idx = np.argmax(f1_scores[:-1]) 
best_threshold = thresholds[best_idx]

print(f"Best Custom Threshold found on Validation Set: {best_threshold:.2f}")

# ---------------------------------------------------------
# STEP 2: EVALUATE THIS THRESHOLD ON THE *TEST* SET
# ---------------------------------------------------------

# Now, use that threshold on the unseen test set
xgb_probs_test = best_xgb_model.predict_proba(X_test_processed)[:, 1]

# Apply the best threshold
optimal_preds_test = (xgb_probs_test >= best_threshold).astype(int)
default_preds_test = best_xgb_model.predict(X_test_processed) # Default 0.5 threshold

# Print the final, honest results
print("\n--- FINAL TEST SET PERFORMANCE ---")
print(f"Default (50%) F1-Score: {f1_score(y_test_processed, default_preds_test):.4f}")
print(f"New Maximum F1-Score:   {f1_score(y_test_processed, optimal_preds_test):.4f}")
print(f"New Precision:          {precision_score(y_test_processed, optimal_preds_test):.4f}")
print(f"New Recall:             {recall_score(y_test_processed, optimal_preds_test):.4f}")

In [ ]:
from sklearn.metrics import average_precision_score

# Get the max F1 score from the validation set for the title
best_f1_val = f1_scores[best_idx]

# =========================================================
# STEP 3: VISUALIZE THE RESULTS
# =========================================================

# Calculate PR Curve data strictly for the TEST set
precision_xgb_test, recall_xgb_test, pr_thresholds_xgb_test = precision_recall_curve(y_test_processed, xgb_probs_test)
auprc_xgb = average_precision_score(y_test_processed, xgb_probs_test)

# Create a side-by-side figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# ---------------------------------------------------------
# Plot 1: Threshold Optimization (Validation Set)
# ---------------------------------------------------------
# Remember to slice precisions and recalls [:-1] to match the thresholds array length
ax1.plot(thresholds, f1_scores[:-1], label='F1-Score', color='#39C5BB', linewidth=3)    
ax1.plot(thresholds, precisions[:-1], label='Precision', color='#FFBACC', linestyle='--') 
ax1.plot(thresholds, recalls[:-1], label='Recall', color='#FFCC11', linestyle='--')      

# Mark the best threshold
ax1.axvline(x=best_threshold, color='39C5BB', linestyle=':', linewidth=2, label=f'Best Threshold ({best_threshold:.2f})')

ax1.set_title(f"Validation Set: XGBoost Threshold Tuning (Max F1: {best_f1_val:.4f})", fontsize=13)
ax1.set_xlabel("Probability Threshold")
ax1.set_ylabel("Score")
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.legend(loc='lower center')
ax1.grid(alpha=0.3)

# ---------------------------------------------------------
# Plot 2: Precision-Recall Curve (Test Set)
# ---------------------------------------------------------
# Plot the main PR curve
ax2.plot(recall_xgb_test, precision_xgb_test, color='#39C5BB', linewidth=3, label=f'Test PR Curve (AUPRC = {auprc_xgb:.3f})')
ax2.fill_between(recall_xgb_test, 0, precision_xgb_test, color='#9467bd', alpha=0.15)

# Find where our best threshold lands on the test curve using np.argmin
idx_xgb = np.argmin(np.abs(pr_thresholds_xgb_test - best_threshold))

# Mark the optimal operating point
ax2.scatter(recall_xgb_test[idx_xgb], precision_xgb_test[idx_xgb], color='red', s=100, zorder=5, label='Optimal Operating Point')

ax2.set_title("Test Set: XGBoost Precision-Recall Curve", fontsize=13)
ax2.set_xlabel("Recall")
ax2.set_ylabel("Precision")
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.legend(loc='lower left')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# SHAP

In [ ]:
import shap

# 1. Initialize the Tree Explainer for XGBoost
explainer_xgb = shap.TreeExplainer(best_xgb_model)
shap_vals_xgb = explainer_xgb.shap_values(X_test_processed)

# XGBoost typically returns a single 2D array for binary classification,
# but we add a safety check just in case your specific version returns a list!
if isinstance(shap_vals_xgb, list):
    shap_raw_xgb = shap_vals_xgb[1]
else:
    shap_raw_xgb = shap_vals_xgb

# Put them in a DataFrame for easy grouping
shap_df_xgb = pd.DataFrame(shap_raw_xgb, columns=X_test_processed.columns)
grouped_shap_df_xgb = shap_df_xgb.copy()

# 2. Define the exact prefixes of your One-Hot Encoded columns
categorical_prefixes = ['cat__Sex','cat__Age_Group','cat__Transmission',
                        'cat__Healthcare_Access_Friction','cat__Civil_Status','cat__OFW_Status','cat__Chemsex_Engagement',
                        'cat__Alcohol_Sex_Risk','cat__PrEP_Awareness','cat__Transactional_Sex','cat__STI_BBV_CoInfection_Any']

# 3. Stitch the dummy columns back together!
for prefix in categorical_prefixes:
    # Find all columns that start with this prefix
    dummy_cols = [col for col in X_test_processed.columns if col.startswith(f"{prefix}_")]

    if len(dummy_cols) > 0:
        # Sum the dummy parts to get the total category importance
        grouped_shap_df_xgb[prefix] = shap_df_xgb[dummy_cols].sum(axis=1)
        # Drop the diluted dummy columns
        grouped_shap_df_xgb.drop(columns=dummy_cols, inplace=True)

In [ ]:
# 4. Plot the Global Feature Importance (Miku Teal!)
plt.figure(figsize=(12, 8))
plt.title(f"XGBoost: True Feature Importance (Grouped)", fontsize=14)

shap.summary_plot(
    grouped_shap_df_xgb.values,
    feature_names=grouped_shap_df_xgb.columns,
    plot_type="bar",
    color="#39C5BB", #turqoise
    show=False
)

plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## TOP 10 INDIVIDUAL FEATURES 

In [ ]:
explainer_xgb = shap.TreeExplainer(best_xgb_model)
shap_vals_xgb = explainer_xgb.shap_values(X_test_processed)

plt.figure(figsize=(10, 6))

shap.summary_plot(
    shap_vals_xgb,
    features=X_test_processed,
    feature_names=X_test_processed.columns,
    plot_type="bar",                      
    color="#39C5BB",
    max_display=10,
    show=False
)

# Optional: Adding a subtle grid
plt.grid(axis='x', color='gray', alpha=0.2, linestyle=':')
plt.tight_layout()
plt.show()


## Beeswarm

In [ ]:
import matplotlib.colors as mcolors
plt.figure(figsize=(10, 6))
miku_cmap = mcolors.LinearSegmentedColormap.from_list("miku_gradient", ["#D1F2F0", "#39C5BB"])
shap.summary_plot(
    shap_vals_xgb,
    features=X_test_processed,
    feature_names=X_test_processed.columns,
    plot_type="dot",                        # Beeswarm style
    cmap=miku_cmap,
    max_display=10,                         # Top 10 only
    show=False
)

# Optional: Adding a subtle grid
plt.grid(axis='x', color='gray', alpha=0.2, linestyle=':')
plt.tight_layout()
plt.show()

# SAVING THE MODEL (RUN MO TO YA IF YOU THINK MAGANDA RESULT PARA MASAVE)

In [ ]:
import joblib

# Save the trained model
joblib.dump(best_xgb_model, 'best_xgb_model.joblib')

# Save the preprocessor so you can transform new data later!
joblib.dump(preprocessor, 'preprocessor.joblib')

print("Model AND Preprocessor successfully saved!")